# Tarea 5 — Física Computacional II
**Nombre completo:** Martina Guerrero Ramos  
**RUT:** 21.888.130-0 
**Fecha:** 28 de octubre de 2026


## Preparación: bibliotecas y catálogo

Importo las bibliotecas necesarias para leer, limpiar, analizar y guardar los datos.
Utilizo el catálogo estelar original proporcionado.

In [38]:
import numpy as np
import pandas as pd
from pathlib import Path
import pickle

# Defino la ruta del catálogo estelar original.
ruta_csv = Path("catalogo_estelar.csv")

## Lectura y limpieza del catálogo

Creo una clase llamada `CatalogoEstelar` para leer y almacenar los datos del catálogo. Luego, realizo una limpieza para eliminar los registros inválidos y los identificadores duplicados, conservando la primera aparición de cada uno.

In [39]:
class CatalogoEstelar:
    def __init__(self, ruta_csv):
        # Leo el catálogo y compruebo que tenga las columnas necesarias.
        datos = pd.read_csv(ruta_csv)
        columnas = ["id", "nombre", "temperatura_K", "luminosidad_Lsol", "distancia_pc"]
        if not set(columnas).issubset(datos.columns):
            raise ValueError("El catálogo no contiene todas las columnas esperadas.")

        # Almaceno cada columna como un arreglo independiente de NumPy.
        self.id = datos["id"].to_numpy(dtype=str)
        self.nombre = datos["nombre"].to_numpy(dtype=str)
        self.temperatura_K = pd.to_numeric(datos["temperatura_K"], errors="coerce").to_numpy()
        self.luminosidad_Lsol = pd.to_numeric(datos["luminosidad_Lsol"], errors="coerce").to_numpy()
        self.distancia_pc = pd.to_numeric(datos["distancia_pc"], errors="coerce").to_numpy()
        self.total_inicial = len(self.id)

        # Ejecuto la limpieza automáticamente.
        self.limpiar_datos()

    def limpiar_datos(self):
        # Identifico los valores finitos y positivos.
        validos = (np.isfinite(self.temperatura_K) &
                   np.isfinite(self.luminosidad_Lsol) &
                   np.isfinite(self.distancia_pc) &
                   (self.temperatura_K > 0) &
                   (self.luminosidad_Lsol > 0) &
                   (self.distancia_pc > 0))

        # Excluyo identificadores y nombres vacíos o ausentes.
        validos &= (~np.isin(np.char.strip(self.id), ["", "nan", "None"]))
        validos &= (~np.isin(np.char.strip(self.nombre), ["", "nan", "None"]))

        # Conservo la primera aparición de cada identificador.
        _, primeros = np.unique(self.id, return_index=True)
        sin_duplicados = np.zeros(len(self.id), dtype=bool)
        sin_duplicados[primeros] = True

        # Aplico la misma máscara a todas las columnas.
        mascara = validos & sin_duplicados
        eliminados = len(self.id) - np.count_nonzero(mascara)

        self.id = self.id[mascara]
        self.nombre = self.nombre[mascara]
        self.temperatura_K = self.temperatura_K[mascara]
        self.luminosidad_Lsol = self.luminosidad_Lsol[mascara]
        self.distancia_pc = self.distancia_pc[mascara]

        print(f"Limpieza: {eliminados} registros eliminados; {len(self.id)} estrellas válidas.")
        return eliminados

    def datos_dict(self):
        # Organizo las columnas en un diccionario.
        return {
            "id": self.id,
            "nombre": self.nombre,
            "temperatura_K": self.temperatura_K,
            "luminosidad_Lsol": self.luminosidad_Lsol,
            "distancia_pc": self.distancia_pc
        }

    def resumen(self):
        # Calculo las propiedades del catálogo limpio con NumPy.
        if len(self.id) == 0:
            raise ValueError("No quedaron estrellas válidas para calcular estadísticas.")

        mas_caliente = np.argmax(self.temperatura_K)
        mas_cercana = np.argmin(self.distancia_pc)

        print("\nRESUMEN DEL CATÁLOGO LIMPIO")
        print("Número de estrellas:", len(self.id))
        print(f"Temperatura media: {np.mean(self.temperatura_K):.2f} K")
        print(f"Temperatura máxima: {np.max(self.temperatura_K):.2f} K")
        print(f"Luminosidad mediana: {np.median(self.luminosidad_Lsol):.2f} Lsol")
        print(f"Distancia mínima: {np.min(self.distancia_pc):.2f} pc")
        print(f"Distancia máxima: {np.max(self.distancia_pc):.2f} pc")
        print("Estrella más caliente:", self.nombre[mas_caliente])
        print("Estrella más cercana:", self.nombre[mas_cercana])

    def guardar_csv(self):
        # Creo la carpeta de resultados y guardo el catálogo en CSV.
        carpeta = Path("resultados")
        carpeta.mkdir(parents=True, exist_ok=True)
        ruta = carpeta / "catalogo copia.csv"

        pd.DataFrame(self.datos_dict()).to_csv(ruta, index=False, encoding="utf-8")
        print("CSV guardado en:", ruta)
        return ruta

    def leer_csv(self):
        # Recupero el CSV utilizando el constructor de la clase.
        return CatalogoEstelar(Path("resultados") / "catalogo copia.csv")

    def guardar_npy(self):
        # Guardo las temperaturas en formato NPY.
        carpeta = Path("resultados")
        carpeta.mkdir(parents=True, exist_ok=True)
        ruta = carpeta / "temperaturas.npy"

        np.save(ruta, self.temperatura_K)
        print("NPY guardado en:", ruta)
        return ruta

    def leer_npy(self):
        return np.load(Path("resultados") / "temperaturas.npy", allow_pickle=False)

    def guardar_npz(self):
        # Guardo todas las columnas en un archivo NPZ comprimido.
        carpeta = Path("resultados")
        carpeta.mkdir(parents=True, exist_ok=True)
        ruta = carpeta / "catalogo completo.npz"

        np.savez_compressed(ruta, **self.datos_dict())
        print("NPZ guardado en:", ruta)
        return ruta

    def leer_npz(self):
        with np.load(Path("resultados") / "catalogo completo.npz", allow_pickle=False) as archivo:
            return {columna: archivo[columna].copy() for columna in archivo.files}

    def guardar_pickle(self):
        # Guardo el diccionario del catálogo en formato Pickle.
        carpeta = Path("resultados")
        carpeta.mkdir(parents=True, exist_ok=True)
        ruta = carpeta / "catalogo completo.pkl"

        with open(ruta, "wb") as archivo:
            pickle.dump(self.datos_dict(), archivo)

        print("Pickle guardado en:", ruta)
        return ruta

    def leer_pickle(self):
        # Recupero el archivo Pickle creado anteriormente.
        with open(Path("resultados") / "catalogo completo.pkl", "rb") as archivo:
            return pickle.load(archivo)

    def verificar_archivos(self):
        # Recupero los archivos y comparo sus datos con el catálogo limpio.
        referencia = self.datos_dict()
        catalogo_csv = self.leer_csv().datos_dict()
        temperaturas_npy = self.leer_npy()
        catalogo_npz = self.leer_npz()
        catalogo_pickle = self.leer_pickle()

        def comparar(datos, columnas):
            if not all(c in datos for c in columnas):
                return False
            if not all(len(datos[c]) == len(self.id) for c in columnas):
                return False

            # Comparo textos con array_equal y números con allclose.
            return all(
                np.array_equal(referencia[c], datos[c])
                if c in ("id", "nombre")
                else np.allclose(referencia[c], datos[c])
                for c in columnas
            )

        resultados = {
            "CSV": comparar(catalogo_csv, referencia),
            "NPY": len(temperaturas_npy) == len(self.id) and
                   np.allclose(self.temperatura_K, temperaturas_npy),
            "NPZ": comparar(catalogo_npz, referencia),
            "Pickle": comparar(catalogo_pickle, referencia)
        }

        print("\nVERIFICACIÓN DE INTEGRIDAD")
        for formato, coincide in resultados.items():
            print(f"{formato}: {'Coincide' if coincide else 'NO coincide'}")

        return resultados

    def comparar_formatos(self):
        # Obtengo el tamaño en bytes de cada archivo.
        carpeta = Path("resultados")
        archivos = {
            "CSV": carpeta / "catalogo copia.csv",
            "NPY": carpeta / "temperaturas.npy",
            "NPZ": carpeta / "catalogo completo.npz",
            "Pickle": carpeta / "catalogo completo.pkl"
        }

        tabla = pd.DataFrame({
            "Formato": list(archivos),
            "Tamaño (bytes)": [
                ruta.stat().st_size for ruta in archivos.values()
            ]
        })

        print("\nCOMPARACIÓN DE FORMATOS")
        print(tabla.to_string(index=False))
        return tabla

### Ejecución y verificación de la limpieza

Ejecuto la limpieza del catálogo y compruebo que, al repetir el procedimiento, no se eliminen más registros, ya que los restantes son válidos.

In [40]:
# Creo el catálogo y ejecuto automáticamente la limpieza.
catalogo = CatalogoEstelar(ruta_csv)

# Compruebo que repetir la limpieza no modifique los registros válidos.
primera_cantidad = len(catalogo.id)
eliminados_segunda_vez = catalogo.limpiar_datos()

assert eliminados_segunda_vez == 0
assert len(catalogo.id) == primera_cantidad

print("Comprobación: la limpieza repetida no cambia los registros válidos.")

Limpieza: 8 registros eliminados; 92 estrellas válidas.
Limpieza: 0 registros eliminados; 92 estrellas válidas.
Comprobación: la limpieza repetida no cambia los registros válidos.


## Análisis con NumPy

Utilizo las funciones de NumPy para calcular las propiedades del catálogo limpio, como las temperaturas, luminosidades y distancias. Además, identifico la estrella con mayor temperatura y la más cercana.

In [41]:
catalogo.resumen()


RESUMEN DEL CATÁLOGO LIMPIO
Número de estrellas: 92
Temperatura media: 6388.60 K
Temperatura máxima: 9829.36 K
Luminosidad mediana: 6.03 Lsol
Distancia mínima: 58.03 pc
Distancia máxima: 490.81 pc
Estrella más caliente: Estrella_0006
Estrella más cercana: Estrella_0039


## Lectura y escritura de archivos

Guardo y recupero los datos del catálogo en los formatos CSV, NPY, NPZ y Pickle. Para ello, creo la carpeta `resultados/` cuando sea necesario.

In [42]:
# Guardo el catálogo en los cuatro formatos.
catalogo.guardar_csv()
catalogo.guardar_npy()
catalogo.guardar_npz()
catalogo.guardar_pickle()

# Recupero los archivos para comprobar su lectura.
copia_csv = catalogo.leer_csv()
temperaturas_recuperadas = catalogo.leer_npy()
copia_npz = catalogo.leer_npz()
copia_pickle = catalogo.leer_pickle()

print("Estrellas en CSV recuperado:", len(copia_csv.id))
print("Temperaturas en NPY recuperado:", len(temperaturas_recuperadas))
print("Columnas en NPZ recuperado:", list(copia_npz))
print("Columnas en Pickle recuperado:", list(copia_pickle))

CSV guardado en: resultados/catalogo copia.csv
NPY guardado en: resultados/temperaturas.npy
NPZ guardado en: resultados/catalogo completo.npz
Pickle guardado en: resultados/catalogo completo.pkl
Limpieza: 0 registros eliminados; 92 estrellas válidas.
Estrellas en CSV recuperado: 92
Temperaturas en NPY recuperado: 92
Columnas en NPZ recuperado: ['id', 'nombre', 'temperatura_K', 'luminosidad_Lsol', 'distancia_pc']
Columnas en Pickle recuperado: ['id', 'nombre', 'temperatura_K', 'luminosidad_Lsol', 'distancia_pc']


## Verificación y comparación de formatos

Compruebo que los archivos guardados conserven la cantidad de estrellas y sus datos originales. Utilizo `np.array_equal()` para comparar los identificadores y nombres, y `np.allclose()` para los valores numéricos. Finalmente, comparo el tamaño de los archivos en bytes.

In [43]:
# Verifico que los archivos recuperados coincidan con el catálogo limpio.
verificacion = catalogo.verificar_archivos()
assert all(verificacion.values()), "Se encontró al menos un archivo con datos diferentes."

# Comparo los tamaños de los archivos en bytes.
tabla_tamanos = catalogo.comparar_formatos()

Limpieza: 0 registros eliminados; 92 estrellas válidas.

VERIFICACIÓN DE INTEGRIDAD
CSV: Coincide
NPY: Coincide
NPZ: Coincide
Pickle: Coincide

COMPARACIÓN DE FORMATOS
Formato  Tamaño (bytes)
    CSV            6953
    NPY             864
    NPZ            3818
 Pickle            9240


### Interpretación de los resultados

**CSV:** Es un formato de texto que permite almacenar los datos con sus encabezados. Es fácil de leer y compartir con distintos programas, aunque no conserva automáticamente los tipos de datos.

**NPY:** Es un formato binario de NumPy que permite guardar un arreglo conservando su estructura y tipo de dato. En este caso, almaceno únicamente las temperaturas.

**NPZ:** Permite guardar varios arreglos NumPy en un solo archivo. Utilizo `np.savez_compressed()` para comprimir los datos y conservar los nombres de las columnas.

**Pickle:** Permite almacenar objetos de Python, como diccionarios. Es útil para guardar los datos del catálogo, aunque resulta menos conveniente para compartirlos con otros programas. Por motivos de seguridad, solo debo abrir archivos Pickle de fuentes confiables.

**¿Qué formato utilizaría para compartir el catálogo?**

Elegiría **CSV**, porque permite compartir todas las columnas y sus encabezados, y puede abrirse fácilmente en distintos programas.

**¿Qué formato utilizaría para almacenar varios arreglos NumPy?**

Elegiría **NPZ comprimido**, porque permite almacenar varios arreglos en un único archivo, conservando sus nombres y tipos de datos.

Al comparar los tamaños obtenidos, observo que NPY ocupa menos espacio, pero esto se debe a que solo almacena las temperaturas y no el catálogo completo.